In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from static_token_div.tools.tools import sigmoid

# Extract data

In [2]:
data_dict = {1: {2: 0, 1: 1, 5: 1, 7: 1},
        3: {4: 1, 1: 0, 58: 1, 17: 1}
}

# Process

In [3]:
d_embedding = 4
vocab_size = 2
eta = 1e-3
nb_iteration = 10

In [4]:
data = []
for target_word, context_dict in data_dict.items():
    for context_word, label in context_dict.items():
        data.append([target_word, context_word, label])

In [5]:
df = pd.DataFrame(data, columns=['target_word', 'context_word', 'label'])

In [19]:
vocab = list(set(df['target_word']).union(set(df['context_word'])))
main_embeddings = pd.DataFrame(np.random.randn(len(vocab), d_embedding), index=vocab)
context_embeddings = pd.DataFrame(np.random.randn(len(vocab), d_embedding), index=vocab)

In [26]:
def update_embeddings_with_loss(df, main_embeddings, context_embeddings, learning_rate, k):
    total_loss = 0
    for _, row in df.iterrows():
        target_word = row['target_word']
        context_word = row['context_word']
        label = row['label']

        # Sélection de k échantillons négatifs
        negative_samples = df['context_word'].sample(k).values

        # Calcul de la perte pour ce batch
        batch_loss = loss_function(target_word, context_word, negative_samples, main_embeddings, context_embeddings)
        total_loss += batch_loss

        # Récupération des vecteurs des mots cibles et de contexte
        m = main_embeddings.loc[target_word].values
        cpos = context_embeddings.loc[context_word].values

        # Mise à jour du vecteur du mot cible
        grad_m = (sigmoid(np.dot(m, cpos)) - 1) * cpos
        main_embeddings.loc[target_word] -= learning_rate * grad_m

        # Mise à jour du vecteur des mots négatifs
        for cneg in negative_samples:
            cneg_vec = context_embeddings.loc[cneg].values
            grad_cneg = sigmoid(np.dot(m, cneg_vec)) * m
            context_embeddings.loc[cneg] -= learning_rate * grad_cneg

        # Mise à jour du vecteur de contexte positif
        grad_cpos = (sigmoid(np.dot(m, cpos)) - 1) * m
        context_embeddings.loc[context_word] -= learning_rate * grad_cpos
    return main_embeddings, context_embeddings, total_loss

In [27]:
def update_embeddings(df, main_embeddings, context_embeddings, learning_rate):
    for _, row in df.iterrows():
        target_word = row['target_word']
        context_word = row['context_word']
        label = row['label']
        
        m = main_embeddings.loc[target_word].values
        c = context_embeddings.loc[context_word].values
        
        z = m @ c
        score = sigmoid(z)
        error = label - score
        
        main_embeddings.loc[target_word] += learning_rate * error * c
        context_embeddings.loc[context_word] += learning_rate * error * m

    return main_embeddings, context_embeddings

In [28]:
for _ in range(nb_iteration):
    main_embeddings, context_embeddings, total_loss = update_embeddings_with_loss(df, main_embeddings, context_embeddings, eta, k=5)
    #main_embeddings, context_embeddings = update_embeddings(df, main_embeddings, context_embeddings, eta)

In [32]:
main_embeddings.shape, context_embeddings.shape, total_loss

((8, 4), (8, 4), np.float64(75.01619036160199))